# Theoretical versus application recoverability

This notebook compares theoretical recoverability with perturbation results. The marginal analysis assigns each theoretical edge to `recovered_by_set` and each application edge to the first intervention set that produced an identifiable result.

In [11]:
from pathlib import Path
import ast
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO = Path.cwd().resolve()
while not (REPO / 'src' / 'nocap').exists() and REPO != REPO.parent:
    REPO = REPO.parent
BASE = REPO / 'notebooks' / 'Ecoli_Analysis_Notebooks'
ESTIMATION_DIR = BASE / 'estimation' / 'csd_n10_k3_umi_paired'
OUTPUT_DIR = BASE / 'estimation' / 'perturbation' / 'theoretical_vs_application_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IDENTIFIABLE_PATH = BASE / 'csd_identifiable_edges.csv'
RECOVERY_PATH = BASE / 'csd_recovery_edges_n10_k3.csv'
RECOVERY_SETS_PATH = BASE / 'csd_recovery_n10_k3.csv'
APPLICATION_DIR = ESTIMATION_DIR / 'csv'

def add_edge_key(frame):
    frame = frame.copy()
    frame['cause'] = frame['cause'].astype('string').str.strip().str.lower()
    frame['effect'] = frame['effect'].astype('string').str.strip().str.lower()
    frame['edge'] = frame['cause'] + '->' + frame['effect']
    return frame

identifiable = add_edge_key(pd.read_csv(IDENTIFIABLE_PATH))
identifiable['adjustment_set'] = identifiable['adjustment_set'].astype('string').fillna('').str.strip()
identifiable = identifiable.loc[identifiable['adjustment_set'].ne('')]
recovery_theory = add_edge_key(pd.read_csv(RECOVERY_PATH))
recovery_theory['recovered'] = recovery_theory['recovered'].astype('string').str.lower().eq('true')
recovery_theory['recovered_by_set'] = pd.to_numeric(recovery_theory['recovered_by_set'], errors='coerce')
recovery_sets = pd.read_csv(RECOVERY_SETS_PATH)
recovery_sets['genes'] = recovery_sets['genes'].map(lambda value: ast.literal_eval(value) if isinstance(value, str) else value)
recovery_sets['genes'] = recovery_sets['genes'].map(lambda genes: [str(g).lower() for g in genes])
recovery_sets['set_label'] = recovery_sets.apply(lambda row: f"set {int(row['set_index'])}: {', '.join(row['genes'])}", axis=1)
set_lookup = recovery_sets.set_index('set_index')['set_label']

frames = []
for path in sorted(APPLICATION_DIR.glob('*.csv')):
    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        continue
    frame['csv_file'] = path.name
    frames.append(frame)
application = add_edge_key(pd.concat(frames, ignore_index=True))
application['status'] = application['status'].astype('string').str.strip().str.lower()
application['run_status'] = application['run_status'].astype('string').str.strip().str.lower()
application['intervention_id'] = application['intervention_id'].astype('string').fillna('').str.strip()
application['adjustment_set'] = application['adjustment_set'].astype('string').fillna('').str.strip()
application['completed'] = application['run_status'].isin(['', 'complete', 'completed'])
application['is_intervention'] = ~application['intervention_id'].str.lower().isin(['', 'observational', 'nan'])
application['application_recovered'] = application['completed'] & application['is_intervention'] & application['status'].eq('identifiable') & application['adjustment_set'].ne('')
print(f'Application rows: {len(application):,}')


Application rows: 2,108,016


In [12]:
theory_identifiable = identifiable[['cause', 'effect', 'edge']].drop_duplicates('edge')
theory_recovered = recovery_theory.loc[recovery_theory['recovered']].copy()
theory_recovered['theoretical_perturbation_sets'] = theory_recovered['recovered_by_set'].map(set_lookup)
theory_recovered = theory_recovered[['cause', 'effect', 'edge', 'theoretical_perturbation_sets']].drop_duplicates('edge')
theoretical = pd.concat([theory_identifiable.assign(theoretical_perturbation_sets=''), theory_recovered], ignore_index=True).groupby(['cause','effect','edge'], as_index=False).agg({'theoretical_perturbation_sets': lambda values: ', '.join(sorted({str(v) for v in values if str(v) not in {'', 'nan', '<NA>'}}))})
theoretical['theoretical_identifiable'] = theoretical['edge'].isin(theory_identifiable['edge'])
theoretical['theoretical_recovery_edge'] = theoretical['edge'].isin(theory_recovered['edge'])
theoretical['theoretical_recoverable'] = theoretical['theoretical_identifiable'] | theoretical['theoretical_recovery_edge']
app_summary = application.groupby(['cause','effect','edge'], as_index=False).agg(application_recoverable=('application_recovered','any'), application_rows=('edge','size'), recovered_application_rows=('application_recovered','sum'), intervention_sets_tested=('intervention_set_index','nunique'))
comparison = theoretical.merge(app_summary, on=['cause','effect','edge'], how='outer')
for column in ['theoretical_identifiable','theoretical_recovery_edge','theoretical_recoverable','application_recoverable']:
    comparison[column] = comparison[column].fillna(False).astype(bool)
comparison['theoretical_only_not_application'] = comparison['theoretical_recoverable'] & ~comparison['application_recoverable']
theoretical_only = comparison.loc[comparison['theoretical_only_not_application']].copy()
comparison.to_csv(OUTPUT_DIR / 'theoretical_vs_application_edge_comparison.csv', index=False)
theoretical_only.to_csv(OUTPUT_DIR / 'theoretical_recoverable_not_application_recovered.csv', index=False)
print(f'Theoretical recoverable: {comparison.theoretical_recoverable.sum():,}')
print(f'Application recoverable: {comparison.application_recoverable.sum():,}')
print(f'Theoretical only: {len(theoretical_only):,}')
display(theoretical_only[['edge','theoretical_perturbation_sets']].head(25))


Theoretical recoverable: 8,948
Application recoverable: 8,761
Theoretical only: 187


,edge,theoretical_perturbation_sets
7,accd->acca,"set 1: gade, gadx, rpod"
15,acrr->mara,"set 1: gade, gadx, rpod"
115,arca->argr,"set 5: argr, dksa, fnr"
163,arca->fnr,"set 5: argr, dksa, fnr"
170,arca->gade,"set 1: gade, gadx, rpod"
171,arca->gadx,"set 1: gade, gadx, rpod"
278,arca->rpos,"set 4: hns, mara, rpos"
385,argr->lrp,"set 1: gade, gadx, rpod"
633,cpxr->mara,"set 1: gade, gadx, rpod"
657,cpxr->rpoe,"set 3: fis, nac, rpoe"


In [13]:
# Marginal gains: theoretical set assignment versus first successful application set.
# Edges already identifiable without perturbation are excluded from both
# marginal series, so set 1 measures incremental perturbation gain.
theory_by_set = recovery_theory.loc[recovery_theory['recovered']].dropna(subset=['recovered_by_set']).drop_duplicates('edge')
theory_edge_set = theory_by_set.set_index('edge')['recovered_by_set'].astype(int)
successful = application.loc[application['application_recovered']].copy()
successful['application_set_index'] = pd.to_numeric(successful['intervention_set_index'], errors='coerce')
application_edge_set = successful.dropna(subset=['application_set_index']).groupby('edge')['application_set_index'].min()
all_edges = comparison.loc[comparison['theoretical_recoverable'], ['edge','theoretical_identifiable']].drop_duplicates('edge').copy()
all_edges['theoretical_set_index'] = all_edges['edge'].map(theory_edge_set).astype('Int64')
all_edges['application_set_index'] = all_edges['edge'].map(application_edge_set).astype('Float64')
all_edges = all_edges.loc[~all_edges['theoretical_identifiable']].copy()
max_set = max(int(recovery_sets['set_index'].max()), int(pd.to_numeric(application['intervention_set_index'], errors='coerce').max()))
rows = []
for set_index in range(1, max_set + 1):
    theory_marginal = int((all_edges['theoretical_set_index'] == set_index).sum())
    app_marginal = int((all_edges['application_set_index'] == set_index).sum())
    rows.append({'set_index': set_index, 'theoretical_marginal_edges': theory_marginal, 'application_marginal_edges': app_marginal, 'marginal_gap_theory_minus_application': theory_marginal - app_marginal, 'theoretical_cumulative_edges': int(all_edges['theoretical_set_index'].le(set_index).sum()), 'application_cumulative_edges': int(all_edges['application_set_index'].le(set_index).sum())})
marginal_summary = pd.DataFrame(rows).merge(recovery_sets[['set_index','set_label','proxy_recovered_new','exact_recovered_new']], on='set_index', how='left')
marginal_summary.to_csv(OUTPUT_DIR / 'perturbation_set_marginal_theory_vs_application.csv', index=False)
display(marginal_summary)


,set_index,theoretical_marginal_edges,application_marginal_edges,marginal_gap_theory_minus_application,theoretical_cumulative_edges,application_cumulative_edges,set_label,proxy_recovered_new,exact_recovered_new
0,1,6418,6167,251,6418,6167,"set 1: gade, gadx, rpod",6358,6418
1,2,35,89,-54,6453,6256,"set 2: fliz, fur, rpoh",52,35
2,3,24,8,16,6477,6264,"set 3: fis, nac, rpoe",31,24
3,4,12,17,-5,6489,6281,"set 4: hns, mara, rpos",17,12
4,5,8,0,8,6497,6281,"set 5: argr, dksa, fnr",11,8
5,6,0,3,-3,6497,6284,"set 6: acca, bglj, cra",6,0
6,7,0,9,-9,6497,6293,"set 7: dinj, exur, galr",6,0
7,8,0,9,-9,6497,6302,"set 8: gutm, higa, hipa",6,0
8,9,2,8,-6,6499,6310,"set 9: ihfa, maze, relb",6,2
9,10,3,7,-4,6502,6317,"set 10: rhar, rpon, rpsf",6,3


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
x = marginal_summary['set_index']
axes[0].bar(x - .18, marginal_summary['theoretical_marginal_edges'], .36, label='Theoretical', color='#176b87')
axes[0].bar(x + .18, marginal_summary['application_marginal_edges'], .36, label='Application', color='#d97706')
axes[0].set(title='Marginal new edges by perturbation set', xlabel='Perturbation set index', ylabel='Newly recovered theoretical edges')
axes[0].set_xticks(x)
axes[0].legend()
axes[1].plot(x, marginal_summary['theoretical_cumulative_edges'], 'o-', label='Theoretical', color='#176b87')
axes[1].plot(x, marginal_summary['application_cumulative_edges'], 'o-', label='Application', color='#d97706')
axes[1].set(title='Cumulative recovered edges', xlabel='Perturbation sets included', ylabel='Cumulative theoretical edges')
axes[1].set_xticks(x)
axes[1].legend()
plot_path = OUTPUT_DIR / 'perturbation_set_marginal_theory_vs_application.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close(fig)
print('Wrote:', plot_path)


Wrote: /Users/colin.pannikkat/projects/nocap.worktrees/32-parameter-estimation-via-single-door-criterion-for-cyclic-graphs/notebooks/Ecoli_Analysis_Notebooks/estimation/perturbation/theoretical_vs_application_outputs/perturbation_set_marginal_theory_vs_application.png


## Interpretation

The theoretical marginal is the number of edges first attributed to each `recovered_by_set`. The application marginal is the number of edges first recovered at that intervention set. A positive gap means the set was theoretically expected to add more edges than it delivered in application.